# Simulated market sessions

We read a parametrization from disk, synthesize order flow from it, fold a book through
that flow, and compute statistics of the book.

Two conventions hold throughout:

- **every series is a step line.** The book holds each state until the next message.
- **a level index means two different things.** The notes count positions on the price
  grid; a LOBSTER file counts prices that carry volume. See
  [`grid-levels-and-lobster-levels.md`](../documentation/grid-levels-and-lobster-levels.md).

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from unito26.lob import config, frames, visualization
from unito26.lob.benchmark import REFERENCE_PRICE
from unito26.lob.messages import BUY, SELL, GridDepth, ReportedDepth
from unito26.lob.orderbook import AXIS_B_VARIANTS, AggregateBook
from unito26.lob.replay import DeltaLog, MarketSession
from unito26.lob.simulate import OrderFlowSimulator

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
pio.templates["unito26"] = go.layout.Template(layout=dict(
    paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
    font=dict(color=INK, size=12),
    xaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
    yaxis=dict(gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)),
))
pio.templates.default = "unito26"

## 1. The parametrization

Event times and types are drawn from a $d$-dimensional Hawkes process with a common
exponential kernel. The conditional intensity of component $i$ is

$$\lambda_i(t) \;=\; \mu_i \;+\; \sum_{j=1}^{d}\alpha_{ij}\sum_{t^j_k<t} e^{-\beta\,(t-t^j_k)},
\qquad \alpha_{ij}\ge 0,\ \beta>0 .$$

The six components are the six ways a message reaches the venue: a market buy or sell, a
limit buy or sell, a withdrawal on either side. Then $\mu_i$ is the rate at which
type-$i$ messages arrive for reasons outside the book, $\alpha_{ij}$ is the jump in that
rate caused by one type-$j$ message — row excited, column exciting — and $\beta$ is the
rate at which the excitation relaxes, common to every pair.

Write $\Gamma = A/\beta$. Then $\Gamma_{ij}$ is the expected number of type-$i$ messages
directly triggered by one of type $j$, and its spectral radius $\rho(\Gamma)$ is the
fraction of all messages that are triggered rather than exogenous. Stationarity requires
$\rho(\Gamma)<1$, and calibrations on exchange data report $0.7$ to $0.9$.

In [ ]:
flow = config.example_order_flow_params()
marks = config.example_mark_params()

print(f"branching ratio      {flow.branching_ratio:.4f}")
print(f"stationary intensity {np.round(flow.stationary_intensity(), 3)}  events/s per type")
print(f"marks                {marks}")
flow.to_frame().set_index(["Component", "Cause"])

At $\rho(\Gamma) = 0.8$, four messages in five arrive because another message did. The
frame is the long form of the parametrization: one row per (excited, exciting) pair, the
baseline on the diagonal and null off it, the decay repeated because it is a scalar.

## 2. Simulating the flow

A book that starts empty is unrepresentative for a while: the first orders have nothing to
trade against. The simulator runs into it before anything is recorded.

In [ ]:
simulator = OrderFlowSimulator(flow, marks, REFERENCE_PRICE, rng=11)
book = AggregateBook()
simulator.warm_up(book, horizon=60.0)
opening = book.copy()          # the state the recorded session starts from
print("after warm-up:", book)

# The stream is a generator that reads the book as it currently stands, so the driver has
# to apply each message before the next is computed.  Consuming it without applying would
# quote every order against the frozen warm-up state.
messages = []
for message in simulator.stream(book, horizon=600.0):
    book.apply(message)
    messages.append(message)
print(f"{len(messages)} messages over 600s")

## 3. Folding the session

`MarketSession` records the top `reported_depth` occupied levels after every message, and
reads the statistics off the book as it goes.

The two depths are separate quantities: `reported_depth` counts occupied levels,
`imbalance_levels` counts grid positions.

In [ ]:
DEPTH = ReportedDepth(10)
LEVELS = tuple(GridDepth(n) for n in (1, 2, 3, 5, 10))
PRICE_UNIT = 100   # LOBSTER quotes dollars x 10000, so a penny tick is 100

session = MarketSession.from_occupied_levels(
    opening.copy(), messages, DEPTH, LEVELS, PRICE_UNIT, online_statistics=True
)
print(f"{len(session.lobster_book)} rows")

# The statistics and the timings use the whole session; the figures draw a window of it.
WINDOW = slice(0, 1500)
session.lobster_book.iloc[:5, :8]

## 4. What came out is what a vendor sells you

`lobster_book` is a LOBSTER orderbook file: $4 \times D$ integer columns, ask price and
size then bid price and size, repeated per level — and **no timestamp**, which in a real
file lives in the row-aligned message file. Time is on the index here.

A side holding fewer than $D$ occupied levels is padded, and the two sentinels have
opposite signs: `-9999999999` on the bid, `+9999999999` on the ask. A filter written for
one lets the other through.

In [ ]:
padded_ask = (session.lobster_book[f"AskPrice{DEPTH}"] == frames.ASK_PADDING).mean()
padded_bid = (session.lobster_book[f"BidPrice{DEPTH}"] == frames.BID_PADDING).mean()
print(f"rows padded at level {DEPTH}:  ask {padded_ask:.1%}   bid {padded_bid:.1%}")

# It validates against the schema, which is what makes "LOBSTER compatible" a fact.
MarketSession.lobster_schema(DEPTH).validate(session.lobster_book).shape

## 5. Order book statistics, computed two ways

The statistics can be read off the evolving book, message by message, or computed from the
finished frame, vectorized and touching no book. They agree everywhere except on $I^n$,
and there only where the frame's reported levels do not span the grid window — which the
session carries in a column of its own.

Timing the two needs care. The fold does two jobs at once, recording the frame and
computing the statistics, so its total is not comparable to a computation alone. Fold the
session twice, once with the statistics and once without: the difference is what the
online statistics cost, and that is the number to set against the vectorized computation.

In [ ]:
def elapsed(call):
    start = time.perf_counter()
    result = call()
    return result, time.perf_counter() - start

_, with_stats = elapsed(lambda: MarketSession.from_occupied_levels(
    opening.copy(), messages, DEPTH, LEVELS, PRICE_UNIT, online_statistics=True))
deferred, frame_only = elapsed(lambda: MarketSession.from_occupied_levels(
    opening.copy(), messages, DEPTH, LEVELS, PRICE_UNIT, online_statistics=False))
from_frame, vectorized = elapsed(deferred.stats_from_frame)

print(f"fold, frame and statistics   {with_stats:7.3f}s")
print(f"fold, frame alone            {frame_only:7.3f}s")
print(f"  statistics, online         {with_stats - frame_only:7.3f}s")
print(f"  statistics, vectorized     {vectorized:7.3f}s")
print(f"  ratio                      {(with_stats - frame_only) / vectorized:7.1f}x")

expected = session.stats.copy()
for n in LEVELS:
    uncovered = ~expected[f"QueueImbalance{n}Covered"].astype(bool)
    expected.loc[uncovered, f"QueueImbalance{n}"] = np.nan
pd.testing.assert_frame_equal(expected, from_frame, check_dtype=False)
print("\nthe two agree")
session.stats[["Spread", "MidPrice", "MicroPrice", "QueueImbalance1", "AskLargestGap"]].head()

## 6. Queue evolution

Pick a level from the dropdown: the top panel is the volume resting at that level on each
side, the bottom panel the price of that level.

In the price panel the two lines cannot cross, and the distance between them is the
$n$-level spread, which widens with depth. Where a side has fewer than $n$ occupied levels
the line breaks: that level did not exist then.

In [ ]:
visualization.level_evolution_figure(session, WINDOW)

The dropdown selects a **reported** level — the $n$-th price carrying volume — because
that is what the frame holds. On a book with holes that is not the $n$-th grid position.

## 7. Mid-price and micro-price

$P^\mu$ lies inside the spread and toward the *thin* side, since it weights each price by
the volume resting on the other one. Written out,

$$P^\mu = P^m + \frac{\phi}{2} I^1,$$

so the micro-price is the mid displaced by the imbalance, in units of the half-spread.

In [ ]:
visualization.touch_figure(session, WINDOW)

## 8. Bid-ask spread and gaps between levels

The gap statistics say how the occupied levels are distributed: `LargestGap` is the longest
run of empty grid positions between two levels that carry volume, over the reported span.
A book can hold a one-tick spread and still be sparse two levels back.

In [ ]:
visualization.gap_figure(session, WINDOW)

## 9. Queue imbalance

$I^n$ sums volume over the first $n$ positions **on the price grid**. Computing it from a
file by summing the first $n$ size *columns* sums the first $n$ prices that carry volume
instead, which is a different statistic: the two windows coincide only where the reported
levels are contiguous.

Neither expression errors, both stay in $[-1,1]$, and both move with the market. The
figure below draws them together; `MarketSession.column_sliced_imbalance` is the second
one, named for what it computes.

In [ ]:
N = GridDepth(5)
visualization.imbalance_figure(session, N, WINDOW).show()

by_column = session.column_sliced_imbalance(N)
difference = (by_column - session.stats[f"QueueImbalance{N}"]).abs()
print(f"they differ on {(difference > 1e-9).mean():.1%} of rows; largest difference {difference.max():.3f}")

## 10. How deep a file do you need?

The price mask can only reach as far as the reported levels span. Write the **grid span**
of a side as the number of grid positions between its best and its deepest reported level;
$I^n$ is recoverable when $n$ is within that span on both sides.

The answer is therefore not a property of the code but of the market: a book with no holes
answers a deep $I^n$ from a shallow file, a sparse one does not.

In [ ]:
sessions_by_depth = {}
for depth in (1, 2, 3, 5, 10):
    trial = MarketSession.from_occupied_levels(
        opening.copy(), messages, ReportedDepth(depth), LEVELS, PRICE_UNIT,
        online_statistics=False,
    )
    trial.stats = trial.stats_from_frame()      # the deferred route, used for what it is
    sessions_by_depth[depth] = trial

visualization.coverage_figure(sessions_by_depth).show()

pd.DataFrame(
    {depth: [1.0 - s.stats[f"QueueImbalance{n}Covered"].mean() for n in LEVELS]
     for depth, s in sessions_by_depth.items()},
    index=pd.Index(LEVELS, name="n"),
).round(3)

## 11. Recording a market session

`from_top_of_book` reads the four touch properties, `from_occupied_levels` asks for the
top $D$ levels, and `DeltaLog` records only what each message changed so that
`from_delta_log` can rebuild the session from the changes alone. All three produce
identical sessions, so the question is which is faster, and that depends on the book
underneath.

The statistics are switched off here. With them on they cost several times what the
recording does, and the table would compare the same computation to itself.

`from_delta_log` does no matching: applying a delta is a write of an absolute volume,
where applying a message is a search plus a match. Against that, the log has to be
recorded first, and the recording is itself a full replay; both numbers are below.

In [ ]:
OPENING_BIDS, OPENING_ASKS = dict(opening.levels_map(BUY)), dict(opening.levels_map(SELL))
SPAN = [m.price for m in messages if m.price < 10 ** 9] + list(OPENING_BIDS) + list(OPENING_ASKS)

# Every variant starts from the same warmed state, sized for the whole run.
def fresh(book_cls):
    book = book_cls.for_prices(SPAN)
    for direction, levels in ((BUY, OPENING_BIDS), (SELL, OPENING_ASKS)):
        for price, volume in levels.items():
            book.set_volume(direction, price, volume)
    return book

sample = messages[:6000]
rows = []
for book_cls in AXIS_B_VARIANTS:
    log, record_cost = elapsed(lambda c=book_cls: DeltaLog.record(fresh(c), sample))
    timings = {}
    for label, call in (
        ("top_of_book", lambda c=book_cls: MarketSession.from_top_of_book(
            fresh(c), sample, LEVELS, PRICE_UNIT, False)),
        ("occupied(1)", lambda c=book_cls: MarketSession.from_occupied_levels(
            fresh(c), sample, ReportedDepth(1), LEVELS, PRICE_UNIT, False)),
        ("occupied(10)", lambda c=book_cls: MarketSession.from_occupied_levels(
            fresh(c), sample, ReportedDepth(10), LEVELS, PRICE_UNIT, False)),
        ("delta_log(10)", lambda c=book_cls, g=log: MarketSession.from_delta_log(
            g, c, ReportedDepth(10), LEVELS, PRICE_UNIT, False)),
    ):
        _, timings[label] = elapsed(call)
    timings["recording the log"] = record_cost
    rows.append(pd.Series(timings, name=book_cls.__name__))

pd.DataFrame(rows).round(3)

Reading the four touch properties costs what asking for one occupied level costs, so the
first two columns are the same measurement twice. Asking for ten costs several times
either, and it costs most on the array-backed book, whose ask-side walk is linear in the
span it crosses.

That is also where rebuilding from the log wins: it reaches ten occupied levels the same
way, but it does so without matching, and on the dict-backed books the matching it avoids
is cheaper than the recording it requires.

## 12. Two optimizations of the fold, measured

The fold does the same work several thousand times a second, so two things about how it is
written show up in the total.

**What it records per message.** `apply` can collect the fills and the level changes it
produced; the dense recorders read neither, so they ask for nothing back.

**Where the rows go.** The clear way to build the frame is a Python list of row lists,
handed to `pd.DataFrame` at the end. The recorders instead write each row in place into a
contiguous array that doubles when full, and hand `pd.DataFrame` the array. The two are
timed in the same two stages — folding, then assembling the frame — because they move the
cost from one stage to the other rather than removing it.

In [ ]:
COLUMNS = frames.lobster_book_columns(DEPTH)

def fold_into_a_list(book_cls, messages, reported_depth, price_unit):
    # The clear version: one Python list per row, one DataFrame at the end.
    book, times, rows = fresh(book_cls), [], []
    for message in messages:
        book.apply(message)
        times.append(message.time)
        rows.append(book.to_lobster_row(price_unit, reported_depth))
    return times, rows

def fold_into_a_buffer(book_cls, messages, reported_depth, price_unit):
    from unito26.lob.replay import _RowBuffer
    book, times = fresh(book_cls), []
    buffer = _RowBuffer(4 * reported_depth, np.int64)
    for message in messages:
        book.apply(message)
        times.append(message.time)
        at = buffer.claim()
        book.write_lobster_row(buffer.array, at, price_unit, reported_depth)
    return times, buffer.finished()

def assemble(times, rows):
    frame = pd.DataFrame(rows, columns=COLUMNS, index=pd.Index(times, name="TimeStamp"))
    return MarketSession.lobster_schema(DEPTH).validate(frame)

def fold_plain(book_cls, messages, record):
    book = fresh(book_cls)
    for message in messages:
        book.apply(message, record=record)
    return book

rows = []
for book_cls in AXIS_B_VARIANTS:
    (times, listed), list_fold = elapsed(
        lambda c=book_cls: fold_into_a_list(c, messages, DEPTH, PRICE_UNIT))
    listed_frame, list_frame = elapsed(lambda: assemble(times, listed))
    (times, buffered), buffer_fold = elapsed(
        lambda c=book_cls: fold_into_a_buffer(c, messages, DEPTH, PRICE_UNIT))
    buffered_frame, buffer_frame = elapsed(lambda: assemble(times, buffered))
    assert listed_frame.equals(buffered_frame)
    _, silent = elapsed(lambda c=book_cls: fold_plain(c, messages, False))
    _, recording = elapsed(lambda c=book_cls: fold_plain(c, messages, True))
    rows.append(pd.Series(
        {"list: fold": list_fold, "list: frame": list_frame, "list: total": list_fold + list_frame,
         "buffer: fold": buffer_fold, "buffer: frame": buffer_frame,
         "buffer: total": buffer_fold + buffer_frame,
         "apply, no result": silent, "apply, recording": recording},
        name=book_cls.__name__,
    ))

pd.DataFrame(rows).round(3)

The buffer does not remove work from the fold — it adds a little, since a row is written
one integer at a time instead of being handed over as a list. What it removes is the frame
assembly, which no longer has to infer a dtype and copy from thousands of separate Python
lists, and that is the larger of the two. The total falls.

Asking `apply` for nothing back is the smaller saving of the two in absolute terms, and
the larger in proportion: the result object costs about twice what the update itself does.

## 13. The gap statistics across the ladder

The dict-backed books walk sorted keys; the bitmap-backed ones read the answer off an
integer, using `measure_largest_binary_gap` and `count_binary_gaps`. Whether that is worth
anything depends on how many levels are occupied — the same conditional the best-price
ladder produced.

In [ ]:
def gap_pass(book, reported_depth):
    for direction in (BUY, SELL):
        book.side_statistics(direction, reported_depth)

rows = []
for regime, mark_params in (("shallow", config.shallow_mark_params()),
                            ("deep", config.deep_mark_params())):
    stream = OrderFlowSimulator(flow, mark_params, REFERENCE_PRICE, rng=3)
    seed_book = AggregateBook()
    stream.warm_up(seed_book, horizon=120.0)
    occupied = len(seed_book.levels_map(BUY)) + len(seed_book.levels_map(SELL))
    timings = {"occupied levels": occupied}
    for book_cls in AXIS_B_VARIANTS:
        book = book_cls.from_levels(dict(seed_book.levels_map(BUY)),
                                    dict(seed_book.levels_map(SELL)))
        start = time.perf_counter()
        for _ in range(2000):
            gap_pass(book, DEPTH)
        timings[book_cls.__name__] = round(time.perf_counter() - start, 3)
    rows.append(pd.Series(timings, name=regime))

pd.DataFrame(rows)

Batching the four statistics into one read of the side has flattened this table. What is
left in it is finding the occupied levels, which every rung has to do; the bit tricks act
only on what happens after that. The ladder still separates the two regimes, and it
separates them by much less than it did when each statistic walked the side for itself.

These numbers are recorded in `dev-context/market-microstructure.md` beside the strand's
other measured findings, including where they contradict the reasoning above.